In [1]:
from mlflow.tracking import MlflowClient


MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

In [2]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/03 - training/experiment_tracking/mlruns/2', creation_time=1775603501922, experiment_id='2', last_update_time=1775603501922, lifecycle_stage='active', name='my-cool-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/03 - training/experiment_tracking/mlruns/1', creation_time=1775520215694, experiment_id='1', last_update_time=1775520215694, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/03 - training/experiment_tracking/mlruns/0', creation_time=1775517629048, experiment_id='0', last_update_time=1775517629048, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [ ]:
client.create_experiment(name="my-cool-experiment")

In [5]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="metrics.rmse < 7",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [6]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 905203bbcf2f4c20881d74de289c6b7e, rmse: 6.3099
run id: 8469035ef211468799bb5689f4378c2e, rmse: 6.3100
run id: 74232f705a104818ad342593d14c9777, rmse: 6.3109
run id: abcc9d5e6b4741d78d93e342f61569e0, rmse: 6.3120
run id: 04b2280d7f074db993c76a3d120bbc60, rmse: 6.3144


In [7]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [42]:
run_id = "9b27100d30ee4fb29a5b286e07497c5b"
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor2")

Registered model 'nyc-taxi-regressor2' already exists. Creating a new version of this model...
2026/04/21 21:54:10 WARNING mlflow.tracking._model_registry.fluent: Run with id 9b27100d30ee4fb29a5b286e07497c5b has no artifacts at artifact path 'model', registering model based on models:/m-6e3bc6393cfd4b58a90eb424e0ec826b instead
2026/04/21 21:54:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-regressor2, version 2
Created version '2' of model 'nyc-taxi-regressor2'.


<ModelVersion: aliases=[], creation_timestamp=1776819250931, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1776819250931, metrics=None, model_id=None, name='nyc-taxi-regressor2', params=None, run_id='9b27100d30ee4fb29a5b286e07497c5b', run_link='', source='models:/m-6e3bc6393cfd4b58a90eb424e0ec826b', status='READY', status_message=None, tags={}, user_id='', version='2', workspace='default'>

In [43]:
model_name = "nyc-taxi-regressor2"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: Production
version: 2, stage: None


C:\Users\Eriton\AppData\Local\Temp\ipykernel_23124\1772384692.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [47]:
model_version = 2
new_stage = "Production"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

C:\Users\Eriton\AppData\Local\Temp\ipykernel_23124\3059153048.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1776819250931, current_stage='Production', deployment_job_state=None, description='', last_updated_timestamp=1776819273648, metrics=None, model_id=None, name='nyc-taxi-regressor2', params=None, run_id='9b27100d30ee4fb29a5b286e07497c5b', run_link='', source='models:/m-6e3bc6393cfd4b58a90eb424e0ec826b', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [48]:
client.set_registered_model_alias(
    name=model_name,
    version=2,
    alias=new_stage
)


In [49]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=['Production'], creation_timestamp=1776819250931, current_stage='Production', deployment_job_state=None, description='The model version 2 was transitioned to Production on 2026-04-21', last_updated_timestamp=1776819279234, metrics=None, model_id=None, name='nyc-taxi-regressor2', params=None, run_id='9b27100d30ee4fb29a5b286e07497c5b', run_link='', source='models:/m-6e3bc6393cfd4b58a90eb424e0ec826b', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [58]:
from sklearn.metrics import mean_squared_error, root_mean_squared_error
import pandas as pd

def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}


In [51]:
df = read_dataframe("data/green_tripdata_2021-02.parquet")

In [52]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'C:\\Users\\Eriton\\source\\repos\\mlops-zoomcamp\\02-model-registry\\experiment_tracking\\preprocessor'

In [53]:
import pickle

with open("preprocessor/preprocessor.bin", "rb") as f_in:
    dv = pickle.load(f_in)

In [54]:
X_test = preprocess(df, dv)

In [55]:
target = "duration"
y_test = df[target].values

In [56]:
model_name

'nyc-taxi-regressor2'

In [59]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")

%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)

CPU times: total: 15.6 ms
Wall time: 82.1 ms


{'rmse': 7.758715202840848}